# Safe Lag Feature Engineering (v3)

**Cải thiện so với v2**: Fix phân phối lệch (distribution shift) của rolling features.

## Vấn đề với v2
- `rolling_mean_28 = shift(1).rolling(28)`: khi forecast Aug 16-31, train có đủ 28 ngày thật;
  nhưng trong forecasting notebook (combined-append), Aug 31 chỉ có 12 ngày thật trong window
  → **distribution shift** → model thấy feature khác hẳn lúc training
- `rolling_std_7`: cùng vấn đề — window co lại khi tiến vào test period

## Giải pháp v3
Xóa shift(1)-based rolling features, thay bằng shift(16)-based (toàn bộ window trong train data):

| Feature xóa | Lý do | Feature thay thế |
|---|---|---|
| `lag_1/2/3/7` | Unsafe cho 9–15/16 ngày test | `lag_16`, `lag_21`, `lag_35` |
| `rolling_mean_7/14` | Window quá ngắn, unsafe | `rolling_mean_30` (shift(16)) |
| `rolling_mean_28` | shift(1) → distribution shift | `rolling_mean_28_safe` (shift(16)) |
| `rolling_std_7` | shift(1) → distribution shift | `rolling_std_28_safe` (shift(16)) |

**Tất cả 6 feature mới**: lookback hoàn toàn trong train data cho toàn bộ Aug 16-31 ✓

**Feature count**: 49 (gốc) − 8 (xóa) + 6 (thêm) = **47 features**

**Kaggle test period: 2017-08-16 → 2017-08-31 (16 ngày)**

## 1. Setup & Load Data

In [1]:
import sys
from pathlib import Path

_here = Path.cwd()
PROJECT_ROOT = next(
    (p for p in [_here] + list(_here.parents) if (p / 'config.py').exists()),
    _here
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from config import PROJECT_ROOT, DATA_DIR, RAW_DIR, PROCESSED_DIR, SPLITS_DIR

MODELS_DIR    = PROJECT_ROOT / 'models'
NOTEBOOKS_DIR = PROJECT_ROOT / 'notebooks'

print(f'PROJECT_ROOT : {PROJECT_ROOT}')
print(f'PROCESSED_DIR: {PROCESSED_DIR}')
print(f'SPLITS_DIR   : {SPLITS_DIR}')

PROJECT_ROOT : D:\Topic_13_Project\Topic_13_Retail_Store_Sales_Time_Series
PROCESSED_DIR: D:\Topic_13_Project\Topic_13_Retail_Store_Sales_Time_Series\data\processed
SPLITS_DIR   : D:\Topic_13_Project\Topic_13_Retail_Store_Sales_Time_Series\data\processed\splits


In [2]:
import pandas as pd
import numpy as np
import shutil

# ── 8 features cần xóa ───────────────────────────────────────────────────────
TOXIC_FEATURES = [
    'lag_1', 'lag_2', 'lag_3', 'lag_7',          # lag ngắn, unsafe 9-15/16 ngày
    'rolling_mean_7', 'rolling_mean_14',           # window ngắn, unsafe
    'rolling_mean_28',                             # shift(1) → distribution shift
    'rolling_std_7',                               # shift(1) → distribution shift
]

# ── Load full raw training data (2013-01-01 to 2017-08-15) ───────────────────
print('Loading train_cleaned.csv ...')
train_raw = pd.read_csv(PROCESSED_DIR / 'cleaned' / 'train_cleaned.csv')
train_raw['date'] = pd.to_datetime(train_raw['date'])
train_raw = train_raw.sort_values(['store_nbr', 'family', 'date']).reset_index(drop=True)

print(f'train_raw : {train_raw.shape}')
print(f'date range: {train_raw["date"].min().date()} → {train_raw["date"].max().date()}')

Loading train_cleaned.csv ...
train_raw : (3000888, 6)
date range: 2013-01-01 → 2017-08-15


## 2. Compute Safe Features

### Safety proof cho Kaggle test (Aug 16–31):
- `lag_N` tại Aug 31 cần Aug (31−N) → phải ≤ Aug 15 → N ≥ 16
- `shift(16).rolling(W)` tại Aug 31: cần Aug 15 → July (16+W−1) ≤ Aug 15 → luôn đúng
- **lag_16, lag_21, lag_35, rolling_mean_30, rolling_mean_28_safe, rolling_std_28_safe**: 100% safe ✓

In [3]:
print('Computing safe lag & rolling features...')

grouped = train_raw.groupby(['store_nbr', 'family'], sort=False)['sales']

# Lag features — tất cả ≥ 16 ngày → safe cho toàn bộ Aug 16-31
for lag in [16, 21, 35]:
    train_raw[f'lag_{lag}'] = grouped.shift(lag)
    print(f'  lag_{lag:<3}: computed')

# rolling_mean_30: shift(16).rolling(30) → mean(sales[d-45]...sales[d-16]) ✓
train_raw['rolling_mean_30'] = grouped.transform(
    lambda x: x.shift(16).rolling(30, min_periods=1).mean()
)

# rolling_mean_28_safe: shift(16).rolling(28) → mean(sales[d-43]...sales[d-16]) ✓
train_raw['rolling_mean_28_safe'] = grouped.transform(
    lambda x: x.shift(16).rolling(28, min_periods=1).mean()
)

# rolling_std_28_safe: shift(16).rolling(28).std → std(sales[d-43]...sales[d-16]) ✓
train_raw['rolling_std_28_safe'] = grouped.transform(
    lambda x: x.shift(16).rolling(28, min_periods=2).std()
)

new_feats_cols = ['lag_16', 'lag_21', 'lag_35',
                  'rolling_mean_30', 'rolling_mean_28_safe', 'rolling_std_28_safe']

print(f'\nNaN summary trên full dataset (expected: chỉ NaN ở đầu mỗi nhóm):')
for col in new_feats_cols:
    n   = train_raw[col].isnull().sum()
    pct = n / len(train_raw) * 100
    print(f'  {col:<25}: {n:>7,} NaN ({pct:.3f}%)')

Computing safe lag & rolling features...
  lag_16 : computed
  lag_21 : computed
  lag_35 : computed

NaN summary trên full dataset (expected: chỉ NaN ở đầu mỗi nhóm):
  lag_16                   :  28,512 NaN (0.950%)
  lag_21                   :  37,422 NaN (1.247%)
  lag_35                   :  62,370 NaN (2.078%)
  rolling_mean_30          :  28,512 NaN (0.950%)
  rolling_mean_28_safe     :  28,512 NaN (0.950%)
  rolling_std_28_safe      :  30,294 NaN (1.010%)


## 3. Xác minh An toàn cho Kaggle Test Period

In [4]:
print('=== VERIFICATION: Internal test (Aug 01–15) ===\n')
test_mask = (train_raw['date'] >= '2017-08-01') & (train_raw['date'] <= '2017-08-15')
test_sub  = train_raw[test_mask]

for col in new_feats_cols:
    n = test_sub[col].isnull().sum()
    status = 'PASS ✓' if n == 0 else f'FAIL ✗ ({n} NaN)'
    print(f'  {col:<25}: {status}')
    assert n == 0, f'BUG: {n} NaN trong {col} ở internal test!'

print()
print('=== Lookback trace cho Kaggle test (Aug 16–31) ===\n')
train_end = pd.Timestamp('2017-08-15')
print(f'{"Date":<10}  {"lag_16 needs":<14}  {"lag_35 needs":<14}  {"rm28_safe needs":<22}  Safe?')
for day in [16, 20, 25, 31]:
    d       = pd.Timestamp(f'2017-08-{day:02d}')
    l16     = d - pd.Timedelta(days=16)
    l35     = d - pd.Timedelta(days=35)
    rm28_to = d - pd.Timedelta(days=16)
    rm28_fr = d - pd.Timedelta(days=43)
    all_ok  = l16 <= train_end and l35 <= train_end and rm28_to <= train_end
    print(
        f'Aug {day:<4}   {l16.strftime("%m-%d"):<14}  {l35.strftime("%m-%d"):<14}  '
        f'{rm28_fr.strftime("%m-%d")} – {rm28_to.strftime("%m-%d"):<10}  {"✓ SAFE" if all_ok else "✗"}'
    )

print('\nAll new features lookback entirely within train data ✓')

=== VERIFICATION: Internal test (Aug 01–15) ===

  lag_16                   : PASS ✓
  lag_21                   : PASS ✓
  lag_35                   : PASS ✓
  rolling_mean_30          : PASS ✓
  rolling_mean_28_safe     : PASS ✓
  rolling_std_28_safe      : PASS ✓

=== Lookback trace cho Kaggle test (Aug 16–31) ===

Date        lag_16 needs    lag_35 needs    rm28_safe needs         Safe?
Aug 16     07-31           07-12           07-04 – 07-31       ✓ SAFE
Aug 20     08-04           07-16           07-08 – 08-04       ✓ SAFE
Aug 25     08-09           07-21           07-13 – 08-09       ✓ SAFE
Aug 31     08-15           07-27           07-19 – 08-15       ✓ SAFE

All new features lookback entirely within train data ✓


## 4. Update Existing Splits

Load splits gốc → xóa 8 toxic → merge 6 safe → lưu vào `splits_safe_lag_v3/`

In [5]:
print(f'Loading splits from: {SPLITS_DIR}')
X_train = pd.read_csv(SPLITS_DIR / 'train_features.csv')
X_val   = pd.read_csv(SPLITS_DIR / 'val_features.csv')
X_test  = pd.read_csv(SPLITS_DIR / 'test_features.csv')

print(f'Original shapes — X_train:{X_train.shape}, X_val:{X_val.shape}, X_test:{X_test.shape}')
print(f'Original features ({X_train.shape[1]}): {X_train.columns.tolist()}')

for col in TOXIC_FEATURES:
    assert col in X_train.columns, f'Toxic feature not in splits: {col}'
print(f'\nVerified: all {len(TOXIC_FEATURES)} toxic features present in splits')

Loading splits from: D:\Topic_13_Project\Topic_13_Retail_Store_Sales_Time_Series\data\processed\splits
Original shapes — X_train:(2918916, 49), X_val:(55242, 49), X_test:(26730, 49)
Original features (49): ['date', 'store_nbr', 'family', 'onpromotion', 'year', 'month', 'day', 'day_of_week', 'week_of_year', 'quarter', 'is_weekend', 'is_month_start', 'is_month_end', 'is_payday', 'lag_1', 'lag_2', 'lag_3', 'lag_7', 'lag_14', 'lag_28', 'lag_364', 'rolling_mean_7', 'rolling_mean_14', 'rolling_mean_28', 'rolling_std_7', 'is_national_holiday', 'is_regional_holiday', 'is_local_holiday', 'is_transferred_holiday', 'holiday_type', 'is_carnaval_feature', 'days_to_next_holiday', 'days_after_last_holiday', 'is_earthquake_period', 'oil_price', 'oil_price_lag_7', 'oil_price_lag_14', 'oil_price_rolling_mean_28', 'oil_price_change_pct', 'type', 'city', 'state', 'store_type_enc', 'city_freq', 'state_freq', 'cluster', 'store_type_encoded', 'store_family', 'store_family_te']

Verified: all 8 toxic features

In [6]:
# ── Xóa 8 toxic features ────────────────────────────────────────────────────
X_train = X_train.drop(columns=TOXIC_FEATURES)
X_val   = X_val.drop(columns=TOXIC_FEATURES)
X_test  = X_test.drop(columns=TOXIC_FEATURES)
print(f'After removing {len(TOXIC_FEATURES)} toxic cols: X_train {X_train.shape}')

# ── Chuẩn bị 6 safe features từ train_raw ────────────────────────────────────
new_feats = train_raw[['date', 'store_nbr', 'family'] + new_feats_cols].copy()
new_feats['date'] = new_feats['date'].dt.strftime('%Y-%m-%d')

# ── Merge vào từng split ─────────────────────────────────────────────────────
row_before = {n: len(df) for n, df in [('train', X_train), ('val', X_val), ('test', X_test)]}

X_train = X_train.merge(new_feats, on=['date', 'store_nbr', 'family'], how='left')
X_val   = X_val.merge(new_feats,   on=['date', 'store_nbr', 'family'], how='left')
X_test  = X_test.merge(new_feats,  on=['date', 'store_nbr', 'family'], how='left')

for name, df in [('train', X_train), ('val', X_val), ('test', X_test)]:
    assert len(df) == row_before[name], f'Row count changed: {name}'

print(f'After adding {len(new_feats_cols)} safe features:')
print(f'  X_train : {X_train.shape}   (expected 47 cols = 49 - 8 + 6)')
print(f'  X_val   : {X_val.shape}')
print(f'  X_test  : {X_test.shape}')

assert X_train.shape[1] == 47, f'Expected 47 cols, got {X_train.shape[1]}'
print(f'\nFeature count assertion passed: 47 cols ✓')

After removing 8 toxic cols: X_train (2918916, 41)
After adding 6 safe features:
  X_train : (2918916, 47)   (expected 47 cols = 49 - 8 + 6)
  X_val   : (55242, 47)
  X_test  : (26730, 47)

Feature count assertion passed: 47 cols ✓


In [7]:
print('=== NaN Verification sau khi merge ===\n')
for split_name, df in [('X_train', X_train), ('X_val', X_val), ('X_test', X_test)]:
    print(f'--- {split_name} ---')
    for col in new_feats_cols:
        n = df[col].isnull().sum()
        status = '✓ 0 NaN' if n == 0 else f'✗ {n} NaN'
        print(f'  {col:<25}: {status}')
    print()

# Critical: X_test (Aug 01-15) phải có 0 NaN trong tất cả safe features
for col in new_feats_cols:
    n = X_test[col].isnull().sum()
    assert n == 0, f'FAIL: {n} NaN trong {col} ở X_test!'

print('All assertions passed ✓')
print(f'\nFull feature list ({X_train.shape[1]} cols):')
for i, col in enumerate(X_train.columns, 1):
    print(f'  {i:2d}. {col}')

=== NaN Verification sau khi merge ===

--- X_train ---
  lag_16                   : ✗ 28512 NaN
  lag_21                   : ✗ 37422 NaN
  lag_35                   : ✗ 62370 NaN
  rolling_mean_30          : ✗ 28512 NaN
  rolling_mean_28_safe     : ✗ 28512 NaN
  rolling_std_28_safe      : ✗ 30294 NaN

--- X_val ---
  lag_16                   : ✓ 0 NaN
  lag_21                   : ✓ 0 NaN
  lag_35                   : ✓ 0 NaN
  rolling_mean_30          : ✓ 0 NaN
  rolling_mean_28_safe     : ✓ 0 NaN
  rolling_std_28_safe      : ✓ 0 NaN

--- X_test ---
  lag_16                   : ✓ 0 NaN
  lag_21                   : ✓ 0 NaN
  lag_35                   : ✓ 0 NaN
  rolling_mean_30          : ✓ 0 NaN
  rolling_mean_28_safe     : ✓ 0 NaN
  rolling_std_28_safe      : ✓ 0 NaN

All assertions passed ✓

Full feature list (47 cols):
   1. date
   2. store_nbr
   3. family
   4. onpromotion
   5. year
   6. month
   7. day
   8. day_of_week
   9. week_of_year
  10. quarter
  11. is_weekend
  12. is_

## 5. Save New Splits (splits_safe_lag_v3/)

In [8]:
SPLITS_V3 = PROCESSED_DIR / 'splits_safe_lag_v3'
SPLITS_V3.mkdir(parents=True, exist_ok=True)

print(f'Saving to: {SPLITS_V3}')

X_train.to_csv(SPLITS_V3 / 'train_features.csv', index=False)
X_val.to_csv(SPLITS_V3   / 'val_features.csv',   index=False)
X_test.to_csv(SPLITS_V3  / 'test_features.csv',  index=False)

for fname in ['train_target.csv', 'val_target.csv', 'val_target_original.csv', 'test_target.csv']:
    shutil.copy(SPLITS_DIR / fname, SPLITS_V3 / fname)
    print(f'  Copied : {fname}')

print(f'\n  Saved  : train_features.csv  {X_train.shape}')
print(f'  Saved  : val_features.csv    {X_val.shape}')
print(f'  Saved  : test_features.csv   {X_test.shape}')

# Verify
for fname, expected in [
    ('train_features.csv', X_train.shape),
    ('val_features.csv',   X_val.shape),
    ('test_features.csv',  X_test.shape),
]:
    df_check = pd.read_csv(SPLITS_V3 / fname, nrows=5)
    assert df_check.shape[1] == expected[1], f'Column mismatch in {fname}'

print('\nSave verification ✓')
print(f'\nSummary:')
print(f'  Removed ({len(TOXIC_FEATURES)}): {TOXIC_FEATURES}')
print(f'  Added   ({len(new_feats_cols)}): {new_feats_cols}')
print(f'  Net change: 49 - 8 + 6 = 47 features')
print(f'  Saved to: {SPLITS_V3}')

Saving to: D:\Topic_13_Project\Topic_13_Retail_Store_Sales_Time_Series\data\processed\splits_safe_lag_v3
  Copied : train_target.csv
  Copied : val_target.csv
  Copied : val_target_original.csv
  Copied : test_target.csv

  Saved  : train_features.csv  (2918916, 47)
  Saved  : val_features.csv    (55242, 47)
  Saved  : test_features.csv   (26730, 47)

Save verification ✓

Summary:
  Removed (8): ['lag_1', 'lag_2', 'lag_3', 'lag_7', 'rolling_mean_7', 'rolling_mean_14', 'rolling_mean_28', 'rolling_std_7']
  Added   (6): ['lag_16', 'lag_21', 'lag_35', 'rolling_mean_30', 'rolling_mean_28_safe', 'rolling_std_28_safe']
  Net change: 49 - 8 + 6 = 47 features
  Saved to: D:\Topic_13_Project\Topic_13_Retail_Store_Sales_Time_Series\data\processed\splits_safe_lag_v3
